# DEH 30-Day PySpark Challenge
## Day 15 — GroupBy & Aggregations

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

products_schema = StructType([
    StructField("product_id",     StringType(), True),
    StructField("product_name",   StringType(), True),
    StructField("category",       StringType(), True),
    StructField("sub_category",   StringType(), True),
    StructField("unit_price",     DoubleType(), True),
    StructField("cost_price",     DoubleType(), True),
    StructField("supplier",       StringType(), True),
    StructField("stock_quantity", IntegerType(),True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")

## Step 5 — groupBy() + agg()

In [ ]:
# Total revenue and order count by region
orders_df.groupBy("region").agg(
    F.count("order_id").alias("order_count"),
    F.round(F.sum("unit_price"), 2).alias("total_revenue"),
    F.round(F.avg("unit_price"), 2).alias("avg_order_value")
).orderBy(F.col("total_revenue").desc()).show()

In [ ]:
# Group by multiple columns
orders_df.groupBy("region", "status").agg(
    F.count("order_id").alias("order_count"),
    F.round(F.sum("unit_price"), 2).alias("total_revenue")
).orderBy("region", "status").show()

## Step 6 — All common aggregation functions

In [ ]:
orders_df.groupBy("region").agg(
    F.count("order_id").alias("order_count"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.round(F.sum("unit_price"), 2).alias("total_revenue"),
    F.round(F.avg("unit_price"), 2).alias("avg_price"),
    F.min("unit_price").alias("min_price"),
    F.max("unit_price").alias("max_price")
).show()

## Step 7 — Filtering after groupBy (HAVING equivalent)

In [ ]:
# Regions with more than 20 orders
orders_df.groupBy("region").agg(
    F.count("order_id").alias("order_count"),
    F.round(F.sum("unit_price"), 2).alias("total_revenue")
).filter(F.col("order_count") > 20) \
 .orderBy(F.col("total_revenue").desc()) \
 .show()

## Step 8 — Real-world pattern: join then groupBy

In [ ]:
# Revenue per customer segment
orders_with_segment = orders_df.join(
    customers_df.select("customer_id", "segment"),
    on="customer_id",
    how="inner"
)

orders_with_segment.groupBy("segment").agg(
    F.count("order_id").alias("order_count"),
    F.round(F.sum("unit_price"), 2).alias("total_revenue"),
    F.round(F.avg("unit_price"), 2).alias("avg_order_value")
).orderBy(F.col("total_revenue").desc()).show()

---
## Assignment — Day 15

---

### Task 1
From `orders.csv`, calculate total revenue, order count, and average order value per `status`.  
Sort by total revenue descending.

In [ ]:
# Task 1 — Write your code here


### Task 2
From `orders.csv`, group by `region` and `payment_method`.  
Calculate order count and total revenue.  
Filter to show only groups with more than 5 orders.

In [ ]:
# Task 2 — Write your code here


### Task 3
Join `orders.csv` with `products.csv` on `product_id`.  
Group by `category` and calculate total revenue, order count, and average quantity.  
Which category has the highest revenue?

In [ ]:
# Task 3 — Write your code here


### Task 4
From `orders.csv`, find the top 5 customers by total spend.  
Join with `customers.csv` to show their full name alongside total spend and order count.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*